In [2]:
# ============================================================
# LIBRERÍAS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import time

In [3]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

GOES_DIR = Path(
    r"C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF"
)

print(GOES_DIR)
print(GOES_DIR.exists())

C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF
True


In [4]:
# ============================================================
# INVENTARIO GENERAL
# ============================================================

files = sorted(
    GOES_DIR.glob("**/*.nc")
)

print(f"Archivos encontrados: {len(files)}")

Archivos encontrados: 168


In [5]:
# ============================================================
# TAMAÑO TOTAL
# ============================================================

size_gb = sum(
    f.stat().st_size
    for f in files
) / 1024**3

print(f"Tamaño total: {size_gb:.2f} GB")

Tamaño total: 46.42 GB


In [6]:
# ============================================================
# INVENTARIO POR DÍA
# ============================================================

days = {}

for f in files:

    day = f.parent.name

    days.setdefault(day, 0)
    days[day] += 1

df_days = pd.DataFrame(
    {
        "date": days.keys(),
        "n_files": days.values()
    }
)

df_days.sort_values(
    "date",
    inplace=True
)

df_days

,date,n_files
1,00,6
2,01,6
3,02,6
4,03,6
5,04,6
0,05,12
6,06,6
7,07,6
8,08,6
9,09,6


In [7]:
# ============================================================
# RESUMEN
# ============================================================

print(df_days.describe())

         n_files
count  24.000000
mean    7.000000
std     2.284161
min     6.000000
25%     6.000000
50%     6.000000
75%     6.000000
max    12.000000


In [8]:
for f in files[:5]:
    print("Archivo:")
    print(f)
    print()

    print("Padre:")
    print(f.parent)
    print()

    print("Abuelo:")
    print(f.parent.parent)
    print("-"*80)

Archivo:
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05\OR_ABI-L2-MCMIPF-M6_G18_s20261110500202_e20261110509510_c20261110509583.nc

Padre:
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05

Abuelo:
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111
--------------------------------------------------------------------------------
Archivo:
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05\OR_ABI-L2-MCMIPF-M6_G18_s20261110510202_e20261110519510_c20261110519587.nc

Padre:
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05

Abuelo:
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111
--------------------------------------------------------------------------------
Archivo:
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05\OR_ABI-L2-MCMIPF-M6_G18_s20261110520202_e20261110529516_c20261110529583.nc

Padre:
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05

Abuelo:
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MC

In [10]:
def extract_oaq_features(ds, ix, iy, window=5):
    """Extrae características multiespectrales (C13, C15, C16) y diferencias de

    temperatura de brillo (BTD) sobre el OAQ.
    """
    if window % 2 == 0:
        raise ValueError("window debe ser impar")

    half = window // 2
    channels = ["CMI_C13", "CMI_C15", "CMI_C16"]
    features = {}

    # 1. Extraer estadísticas para cada canal de interés
    for ch in channels:
        # Recorte optimizado en xarray para el canal actual
        subset = ds[ch].isel(
            y=slice(iy - half, iy + half + 1),
            x=slice(ix - half, ix + half + 1),
        )

        prefix = ch.lower()
        features[f"{prefix}_mean"] = float(subset.mean().item())
        features[f"{prefix}_std"] = float(subset.std().item())
        features[f"{prefix}_min"] = float(subset.min().item())
        features[f"{prefix}_max"] = float(subset.max().item())

    # 2. Calcular Diferencias de Temperatura de Brillo (BTD) para análisis de nubes
    features["btd_13_15"] = (
        features["cmi_c13_mean"] - features["cmi_c15_mean"]
    )
    features["btd_13_16"] = (
        features["cmi_c13_mean"] - features["cmi_c16_mean"]
    )
    features["btd_15_16"] = (
        features["cmi_c15_mean"] - features["cmi_c16_mean"]
    )

    return features

In [11]:
def process_goes_file(file_path, ix, iy, window=5):
    """Abre un archivo GOES local, extrae las métricas multiespectrales sobre el

    OAQ y añade la marca de tiempo (timestamp).
    """
    # Usamos 'with' para asegurar que el archivo se cierre correctamente después de leerlo
    with xr.open_dataset(file_path, engine="h5netcdf") as ds:
        # Extraer las variables calculadas (C13, C15, C16 y BTDs)
        features = extract_oaq_features(ds, ix, iy, window)

        # Extraer y convertir el tiempo central de la observación satelital
        timestamp = pd.to_datetime(ds.t.values)
        features["timestamp"] = timestamp

    return features

In [14]:
import xarray as xr

ds = xr.open_dataset(
    files[0],
    engine="h5netcdf"
)

In [15]:
from pyproj import Proj

sat_h = ds.goes_imager_projection.perspective_point_height
sat_lon = ds.goes_imager_projection.longitude_of_projection_origin
sat_sweep = ds.goes_imager_projection.sweep_angle_axis

In [16]:
lon_oaq = -78.5025
lat_oaq = -0.2150

In [17]:
p = Proj(
    proj="geos",
    h=sat_h,
    lon_0=sat_lon,
    sweep=sat_sweep
)

x_oaq, y_oaq = p(
    lon_oaq,
    lat_oaq
)

In [18]:
sat_h_val = sat_h.item()

x_scan = x_oaq / sat_h_val
y_scan = y_oaq / sat_h_val

In [19]:
import numpy as np

ix = np.abs(
    ds.x.values - x_scan
).argmin()

iy = np.abs(
    ds.y.values - y_scan
).argmin()

print(ix, iy)

5196 2722


In [20]:
ds.close()

In [21]:
import time

test_file = files[0]

inicio = time.time()

resultado = process_goes_file(
    test_file,
    ix,
    iy,
    window=5
)

fin = time.time()

print(resultado)
print()
print(f"Tiempo: {fin-inicio:.2f} s")

{'cmi_c13_mean': 261.90069580078125, 'cmi_c13_std': 14.06225299835205, 'cmi_c13_min': 228.01287841796875, 'cmi_c13_max': 273.7341613769531, 'cmi_c15_mean': 259.5164794921875, 'cmi_c15_std': 13.830950736999512, 'cmi_c15_min': 226.26962280273438, 'cmi_c15_max': 271.7145080566406, 'cmi_c16_mean': 250.0569305419922, 'cmi_c16_std': 10.701308250427246, 'cmi_c16_min': 224.12452697753906, 'cmi_c16_max': 258.77081298828125, 'btd_13_15': 2.38421630859375, 'btd_13_16': 11.843765258789062, 'btd_15_16': 9.459548950195312, 'timestamp': Timestamp('2026-04-21 05:05:05.671870976')}

Tiempo: 0.25 s


In [23]:
for i, f in enumerate(files[:10]):
    print(i, f)

0 C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05\OR_ABI-L2-MCMIPF-M6_G18_s20261110500202_e20261110509510_c20261110509583.nc
1 C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05\OR_ABI-L2-MCMIPF-M6_G18_s20261110510202_e20261110519510_c20261110519587.nc
2 C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05\OR_ABI-L2-MCMIPF-M6_G18_s20261110520202_e20261110529516_c20261110529583.nc
3 C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05\OR_ABI-L2-MCMIPF-M6_G18_s20261110530202_e20261110539523_c20261110539581.nc
4 C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05\OR_ABI-L2-MCMIPF-M6_G18_s20261110540202_e20261110549511_c20261110549579.nc
5 C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\111\05\OR_ABI-L2-MCMIPF-M6_G18_s20261110550202_e20261110559516_c20261110559586.nc
6 C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\130\00\OR_ABI-L2-MCMIPF-M6_G18_s20261300000223_e20261300009543_c20261300010007.nc
7 C:\Users\Brian OAQ\data\n

In [24]:
for i, f in enumerate(files[:10]):

    try:
        ds = xr.open_dataset(
            f,
            engine="h5netcdf"
        )

        ds.close()

        print(i, "OK")

    except Exception as e:

        print(i, "ERROR")
        print(f)
        print(e)
        print()

0 OK
1 OK
2 OK
3 OK
4 OK
5 OK
6 ERROR
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\130\00\OR_ABI-L2-MCMIPF-M6_G18_s20261300000223_e20261300009543_c20261300010007.nc
Unable to synchronously open file (file signature not found)

7 ERROR
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\130\00\OR_ABI-L2-MCMIPF-M6_G18_s20261300010223_e20261300019538_c20261300020004.nc
Unable to synchronously open file (file signature not found)

8 ERROR
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\130\00\OR_ABI-L2-MCMIPF-M6_G18_s20261300020223_e20261300029532_c20261300030003.nc
Unable to synchronously open file (file signature not found)

9 ERROR
C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF\2026\130\00\OR_ABI-L2-MCMIPF-M6_G18_s20261300030223_e20261300039544_c20261300040005.nc
Unable to synchronously open file (file signature not found)



In [25]:
for f in files[6:10]:

    print(f.name)
    print(
        f.stat().st_size / 1024**2,
        "MB"
    )
    print()

OR_ABI-L2-MCMIPF-M6_G18_s20261300000223_e20261300009543_c20261300010007.nc
330.575891494751 MB

OR_ABI-L2-MCMIPF-M6_G18_s20261300010223_e20261300019538_c20261300020004.nc
328.1007709503174 MB

OR_ABI-L2-MCMIPF-M6_G18_s20261300020223_e20261300029532_c20261300030003.nc
325.58922386169434 MB

OR_ABI-L2-MCMIPF-M6_G18_s20261300030223_e20261300039544_c20261300040005.nc
323.0972547531128 MB



In [26]:
for f in files[:5]:

    print(f.name)
    print(
        f.stat().st_size / 1024**2,
        "MB"
    )

OR_ABI-L2-MCMIPF-M6_G18_s20261110500202_e20261110509510_c20261110509583.nc
250.9557819366455 MB
OR_ABI-L2-MCMIPF-M6_G18_s20261110510202_e20261110519510_c20261110519587.nc
248.80004596710205 MB
OR_ABI-L2-MCMIPF-M6_G18_s20261110520202_e20261110529516_c20261110529583.nc
246.76759719848633 MB
OR_ABI-L2-MCMIPF-M6_G18_s20261110530202_e20261110539523_c20261110539581.nc
244.5825653076172 MB
OR_ABI-L2-MCMIPF-M6_G18_s20261110540202_e20261110549511_c20261110549579.nc
242.52825164794922 MB


In [22]:
inicio = time.time()

for f in files[:10]:

    _ = process_goes_file(
        f,
        ix,
        iy,
        window=5
    )

fin = time.time()

print(f"Tiempo total: {fin-inicio:.2f} s")
print(f"Tiempo medio: {(fin-inicio)/10:.2f} s")

OSError: Unable to synchronously open file (file signature not found)

In [27]:
def safe_process_goes_file(
    file_path,
    ix,
    iy,
    window=5
):

    try:

        return process_goes_file(
            file_path,
            ix,
            iy,
            window
        )

    except Exception as e:

        print(
            f"ERROR: {file_path.name}"
        )

        return None

In [28]:
resultados = []

for f in files:

    row = safe_process_goes_file(
        f,
        ix,
        iy,
        window=5
    )

    if row is not None:
        resultados.append(row)

print(len(resultados))

ERROR: OR_ABI-L2-MCMIPF-M6_G18_s20261300000223_e20261300009543_c20261300010007.nc
ERROR: OR_ABI-L2-MCMIPF-M6_G18_s20261300010223_e20261300019538_c20261300020004.nc
ERROR: OR_ABI-L2-MCMIPF-M6_G18_s20261300020223_e20261300029532_c20261300030003.nc
ERROR: OR_ABI-L2-MCMIPF-M6_G18_s20261300030223_e20261300039544_c20261300040005.nc
ERROR: OR_ABI-L2-MCMIPF-M6_G18_s20261300040223_e20261300049544_c20261300050005.nc
ERROR: OR_ABI-L2-MCMIPF-M6_G18_s20261300050223_e20261300059543_c20261300100001.nc
ERROR: OR_ABI-L2-MCMIPF-M6_G18_s20261300100223_e20261300109544_c20261300110006.nc
ERROR: OR_ABI-L2-MCMIPF-M6_G18_s20261300110223_e20261300119532_c20261300120000.nc
ERROR: OR_ABI-L2-MCMIPF-M6_G18_s20261300120223_e20261300129532_c20261300130001.nc
ERROR: OR_ABI-L2-MCMIPF-M6_G18_s20261300130223_e20261300139544_c20261300140005.nc
158
